# ModelistsGCN Tutorial

Author: Noa Konforti, Tal Goldberg*, Michal Danino*, Shahar Alon

**End-to-End Pipeline Usage**: run the full pipeline with a single `run(cfg)` call.


## 1. Installation

This section shows how to install **ModelistsGCN** and verify the installation.

We provide two installation options:

1. **Install from GitHub (local install)** – clone/download the repository and install locally  
2. **Install from PyPI** – install directly using `pip`

### 1.1 GitHub
This option is best if you want to:
- edit the code
- develop the package
- run the newest version from the repository

Steps:
1. Download/clone the repository
2. Install locally using `pip install -e .`

In [ ]:
!git clone https://github.com/NoaKonforti/ModelistsGCN
%cd ModelistsGCN_package
!pip install -e .

#### If you *downloaded a ZIP* from GitHub (instead of git clone)

1. Download ZIP  
2. Extract it  
3. In a terminal, `cd` into the extracted folder (where `pyproject.toml` exists)  
4. Run:`pip install -e .`

### 1.2 PyPI
This option is best if you want the stable release without editing code.


In [2]:
!pip install -U ModelistsGCN

### 1.3 Verify installation

In [3]:
def import_modelistsgcn():
    # Try lowercase import first (common PyPI style)
    try:
        import modelistsgcn as pkg
        return pkg, "modelistsgcn"
    except Exception as e1:
        # Try CamelCase import (common in academic repos)
        try:
            import ModelistsGCN as pkg
            return pkg, "ModelistsGCN"
        except Exception as e2:
            raise ImportError(
                "Could not import either 'modelistsgcn' or 'ModelistsGCN'.\n"
                f"Lowercase import error: {e1}\n"
                f"CamelCase import error: {e2}\n"
                "Fix: check your package folder name under src/ and your pyproject.toml."
            )

pkg, import_name = import_modelistsgcn()
print("Imported as:", import_name)
print("Version:", getattr(pkg, "__version__", "NO __version__ FOUND"))
print("Has run():", hasattr(pkg, "run"))
print("run =", getattr(pkg, "run", None))


ImportError: Could not import either 'modelistsgcn' or 'ModelistsGCN'.
Lowercase import error: No module named 'modelistsgcn'
CamelCase import error: No module named 'ModelistsGCN'
Fix: check your package folder name under src/ and your pyproject.toml.

## 2. Configuration ('cfg')
ModelistsGCN uses a configuration dictionary `cfg` that defines required inputs and key parameters.

#### Required inputs

The configuration must include:

- **`expr_csv`**  
  Path to a gene expression matrix (cells × genes), indexed by `cellID`.


- **`num_clusters`**  
  The number of clusters (K) to infer.


- **`markers_csv`**  
  A table of marker genes per cell type (used to define modelist anchor cells).

---

### Morphology inputs (two alternative modes)

ModelistsGCN supports **two mutually exclusive modes** for incorporating morphological features:

---

#### **Option A — Automatic morphology extraction (from segmentation)**

Provide:
- **`segmentation_npy`**  
  A 3D segmentation mask (`.npy`) where each voxel contains a cell label.

In this mode, the pipeline will:
- Extract per-cell morphology features (e.g., volume, surface area, shape descriptors)
- Compute cell centroids automatically

**Important:**  
When using `segmentation_npy`, ensure that the following parameters in the `cfg["features"]` section are correctly defined:
- **`voxel_size`** — physical size of each voxel (e.g., in µm)  
- **`factor_exp`** — expansion factor (relevant for ExSeq data)
---

#### **Option B — Precomputed morphology features**

Provide:
- **`morpho_features_csv`**  
  A CSV file containing per-cell morphological features  
  (must include a `cellID` column)

- **`centroids_csv`**  
  A CSV file with spatial coordinates in the format:
  cellID, centroid_z, centroid_y, centroid_x
  
In this mode:
- No morphology computation is performed
- Features and centroids are loaded directly

---

### Important notes

- Exactly **one morphology mode must be used**:
- either `segmentation_npy`  
- or (`morpho_features_csv` + `centroids_csv`)

- All inputs must be aligned by **`cellID`** to ensure correct integration across modalities.

In [4]:
from pathlib import Path

cfg = {
    "expression_csv": "./tutorial_data/expression.csv",
    "segmentation_npy": "./tutorial_data/segmentation.npy",
    "markers_csv": "./tutorial_data/markers.csv",
    "morpho_features_csv":None,
    "centroids_csv":None,
    "num_clusters": 8,
    "return_model":True,

    "modelists": {
        "quantile_thresh": 0.9,
        "min_modelists": 7,
        "min_types_with_modelists":2,
        "min_gene_fraction": 0.9,
        "max_other_gene_fraction": 0.1,       
    },
    
    "features": {
        "voxel_size":(1.5, 1.0, 1.0),
        "cell_thresh" : 20,
        "factor_exp": 1,
    },
    "graph":{},

    "training": {
        "alpha": 1.0,
        "beta": 0.6,
        "gamma": 1.0,
        "epochs": 20,
        "lr": 1e-2,
        "seed": 5507
    }
}

## 3. Run ModelistsGCN

### End-to-End pipeline usage

**Input**: `cfg`  
**Output**:
- `model`: trained ModelistsGCN model object
- `pred_df`: DataFrame with index `cellID` and column `pred` (cluster assignment)

In [5]:
from ModelistsGCN.pipeline import run

model, pred = run(cfg)

ModuleNotFoundError: No module named 'ModelistsGCN'